In [22]:
import pandas as pd
import scanpy as sc
import muon as mu
from anndata import AnnData

adt_df = pd.read_csv(
    "/mnt/data/anis/data/real_data/teaseq/GSM4949911_tea_fulldepth_adt_counts.csv.gz",
    compression="gzip",
    index_col=0,
)
adata = sc.read_10x_h5(
    "/mnt/data/anis/data/real_data/teaseq/GSM4949911_X061-AP0C1W1_leukopak_perm-cells_tea_fulldepth_cellranger-arc_filtered_feature_bc_matrix.h5",
    gex_only=False,
)

# Split RNA and ATAC
rna = adata[:, adata.var["feature_types"] == "Gene Expression"].copy()
atac = adata[:, adata.var["feature_types"] == "Peaks"].copy()
rna.obs_names = rna.obs_names.str.replace("-1$", "", regex=True)
atac.obs_names = atac.obs_names.str.replace("-1$", "", regex=True)
adt_df.index = adt_df.index.str.replace("-1$", "", regex=True)

# Match cells across modalities
common_cells = rna.obs_names.intersection(atac.obs_names).intersection(adt_df.index)

rna = rna[common_cells].copy()
atac = atac[common_cells].copy()
adt_df = adt_df.loc[common_cells]

# Create ADT AnnData
adt = AnnData(
    X=adt_df.values,
    obs=pd.DataFrame(index=adt_df.index),
    var=pd.DataFrame(index=adt_df.columns),
)
adt.var["feature_types"] = "Protein Expression"
rna.var_names_make_unique()
atac.var_names_make_unique()
adt.var_names_make_unique()
# Create MuData
mdata = mu.MuData(
    {
        "rna": rna,
        "atac": atac,
        "adt": adt,
    }
)

print(mdata)

# mdata.write_h5mu("/mnt/data/anis/data/real_data/teaseq/processed.h5mu")


/mnt/data/anis/miniconda3/envs/phare-env-v2/lib/python3.11/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/mnt/data/anis/miniconda3/envs/phare-env-v2/lib/python3.11/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/mnt/data/anis/miniconda3/envs/phare-env-v2/lib/python3.11/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


MuData object with n_obs × n_vars = 8213 × 103476
  3 modalities
    rna:	8213 × 36601
      var:	'gene_ids', 'feature_types', 'genome', 'interval'
    atac:	8213 × 66828
      var:	'gene_ids', 'feature_types', 'genome', 'interval'
    adt:	8213 × 47
      var:	'feature_types'


/mnt/data/anis/miniconda3/envs/phare-env-v2/lib/python3.11/site-packages/mudata/_core/mudata.py:1416: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/mnt/data/anis/miniconda3/envs/phare-env-v2/lib/python3.11/site-packages/mudata/_core/mudata.py:565: UserWarning: Cannot join columns with the same name because var_names are intersecting.
  self._update_attr_legacy(attr, axis, join_common, **kwargs)
/mnt/data/anis/miniconda3/envs/phare-env-v2/lib/python3.11/site-packages/mudata/_core/mudata.py:1272: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which wi

In [2]:
import os

import muon as mu
import scanpy as sc

adata_paths = [
    "/mnt/data/anis/data/real_data/Pbmc10k/RNA.h5ad",
    "/mnt/data/anis/data/real_data/Pbmc10k/ATAC.h5ad",
]

modalities = {}

for path in adata_paths:
    modality = os.path.splitext(os.path.basename(path))[0].lower()

    if path.endswith(".h5ad"):
        adata = sc.read_h5ad(path)
    elif path.endswith(".h5"):
        adata = sc.read_10x_h5(path)
    else:
        raise ValueError(f"Unsupported file format: {path}")

    modalities[modality] = adata


mdata = mu.MuData(modalities)

print(mdata)
# mdata.write_h5mu("/mnt/data/anis/data/real_data/Pbmc10k/processed.h5mu")

MuData object with n_obs × n_vars = 9631 × 136289
  var:	'feature_types'
  2 modalities
    rna:	9631 × 29095
      obs:	'domain', 'cell_type'
      var:	'gene_ids', 'feature_types'
    atac:	9631 × 107194
      obs:	'domain', 'cell_type'
      var:	'feature_types'
      uns:	'spectral_eigenvalue'
      obsm:	'X_spectral', 'X_umap'


/mnt/data/anis/miniconda3/envs/phare-env-v2/lib/python3.11/site-packages/mudata/_core/mudata.py:1416: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/mnt/data/anis/miniconda3/envs/phare-env-v2/lib/python3.11/site-packages/mudata/_core/mudata.py:1272: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)
